# Phase 5 – Feature Engineering
## Daily Electricity Consumption Prediction in Smart Buildings (BDG2)

**Goal:** Demonstrate the engineered features, verify leakage prevention, and visualise feature relationships.

The heavy computation (lag/rolling for 1,440 buildings) was run by `src/feature_engineering.py`.
This notebook **loads the saved outputs** and produces analysis plots.

**Output file:** `data/processed/features_dataset.parquet` — 994,812 rows × 40 columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path('..').resolve()
GRAPHS_DIR   = PROJECT_ROOT / 'results' / 'graphs'
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print(f'pandas  {pd.__version__}  |  numpy  {np.__version__}')

## 1. Load Feature Dataset

In [ ]:
PARQ = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset.parquet'
df = pd.read_parquet(PARQ)
df['date'] = pd.to_datetime(df['date'])

print(f'Shape      : {df.shape}')
print(f'Buildings  : {df["building_id"].nunique():,}')
print(f'Date range : {df["date"].min().date()}  to  {df["date"].max().date()}')
print(f'NaN target : {df["meter_reading_daily"].isna().sum()}')
print(f'NaN lag_1  : {df["lag_1_day"].isna().sum()}')
df.head(3)

## 2. Feature Groups

In [ ]:
summary = pd.read_csv(PROJECT_ROOT / 'results' / 'tables' / '04_feature_summary.csv')
cat_counts = summary.groupby('Category').size().reset_index(name='Feature Count')
print('Features by category:')
print(cat_counts.to_string(index=False))
print(f'\nTotal features: {len(summary)}')

## 3. Leakage Verification

**Rule:** `lag_1_day` on date D must equal `meter_reading_daily` on date D-1 (for the same building).

In [ ]:
for bldg in df['building_id'].unique()[:3]:
    bdf = df[df['building_id'] == bldg].sort_values('date').reset_index(drop=True)
    expected = bdf['meter_reading_daily'].shift(1)
    mismatch = (abs(bdf['lag_1_day'] - expected) > 0.001).sum()
    status = '✅ PASSED' if mismatch == 0 else f'❌ FAILED ({mismatch} mismatches)'
    print(f'{bldg[:40]:40s}  {status}')

print('\nSample (5 rows) for first building:')
bdf2 = df[df['building_id'] == df['building_id'].iloc[0]].sort_values('date')
print(bdf2[['date','meter_reading_daily','lag_1_day','lag_7_days','rolling_7d_mean']].head(5).to_string(index=False))

## 4. Lag Feature Visualisation
Plot actual consumption vs lag_1_day vs rolling_7d_mean for 60 days on one building.

In [ ]:
bldg = df['building_id'].iloc[0]
bdf  = df[df['building_id'] == bldg].sort_values('date').head(60)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(bdf['date'], bdf['meter_reading_daily'],  label='Actual (Day D)',      lw=2,   color='steelblue')
ax.plot(bdf['date'], bdf['lag_1_day'],             label='lag_1_day (Day D-1)', lw=1.5, ls='--', color='darkorange')
ax.plot(bdf['date'], bdf['rolling_7d_mean'],       label='rolling_7d_mean',     lw=1.5, ls=':',  color='green')
ax.set_title(f'Lag Feature Verification — {bldg}', fontsize=12, fontweight='bold')
ax.set_xlabel('Date');  ax.set_ylabel('kWh/day')
ax.legend();  ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / '10_lag_feature_verification.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/10_lag_feature_verification.png')

## 5. Calendar Features vs Consumption

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Weekday
day_means = df.groupby('day_of_week')['meter_reading_daily'].median()
axes[0].bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], day_means.values,
            color=['steelblue']*5+['coral']*2)
axes[0].set_title('Median Consumption by Weekday');  axes[0].set_ylabel('kWh/day')
axes[0].grid(axis='y', alpha=0.3)

# Month
month_means = df.groupby('month')['meter_reading_daily'].median()
axes[1].bar(range(1,13), month_means.values, color='steelblue')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].set_title('Median Consumption by Month');  axes[1].set_ylabel('kWh/day')
axes[1].grid(axis='y', alpha=0.3)

# Season
season_means = df.groupby('season')['meter_reading_daily'].median()
axes[2].bar(['Winter','Spring','Summer','Autumn'], season_means.values,
            color=['#4878CF','#6ACC65','#D65F5F','#B47CC7'])
axes[2].set_title('Median Consumption by Season');  axes[2].set_ylabel('kWh/day')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Calendar Features vs Energy Consumption', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / '11_calendar_feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/11_calendar_feature_distributions.png')

## 6. Derived Weather Features vs Consumption

In [ ]:
sample = df.sample(n=5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(sample['cooling_degree_day'], np.log1p(sample['meter_reading_daily']),
                alpha=0.3, s=5, color='steelblue')
axes[0].set_xlabel('Cooling Degree Day (°C above 18°C)')
axes[0].set_ylabel('log(1 + kWh/day)')
axes[0].set_title('Cooling Degree Day vs Log-Consumption')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(sample['temp_range'], np.log1p(sample['meter_reading_daily']),
                alpha=0.3, s=5, color='coral')
axes[1].set_xlabel('Daily Temperature Range (°C)')
axes[1].set_ylabel('log(1 + kWh/day)')
axes[1].set_title('Temperature Range vs Log-Consumption')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Derived Weather Features vs Energy Consumption', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / '12_derived_weather_vs_consumption.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/12_derived_weather_vs_consumption.png')

## 7. Lag Feature Scatter — How Predictive Are Lags?

In [ ]:
sample = df.sample(n=8000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(np.log1p(sample['lag_1_day']), np.log1p(sample['meter_reading_daily']),
                alpha=0.2, s=4, color='steelblue')
axes[0].set_xlabel('log(1 + lag_1_day)');  axes[0].set_ylabel('log(1 + Actual kWh/day)')
axes[0].set_title('Lag 1 Day vs Actual');  axes[0].grid(True, alpha=0.3)

axes[1].scatter(np.log1p(sample['rolling_7d_mean']), np.log1p(sample['meter_reading_daily']),
                alpha=0.2, s=4, color='green')
axes[1].set_xlabel('log(1 + rolling_7d_mean)');  axes[1].set_ylabel('log(1 + Actual kWh/day)')
axes[1].set_title('7-Day Rolling Mean vs Actual');  axes[1].grid(True, alpha=0.3)

plt.suptitle('Lag Feature Strength (log scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / '13_lag_feature_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/13_lag_feature_scatter.png')

## 8. Final Feature Set Summary

In [ ]:
id_cols  = ['building_id','site_id','date']
target   = ['meter_reading_daily']
calendar = ['day_of_week','is_weekend','month','season','day_of_year','week_of_year']
lag_roll = ['lag_1_day','lag_7_days','rolling_7d_mean','rolling_14d_mean','rolling_7d_std']
weather  = ['air_temperature_mean','air_temperature_min','air_temperature_max',
            'dew_temperature_mean','wind_speed_mean','precip_depth_1hr_total',
            'sea_lvl_pressure_mean','cooling_degree_day','temp_range']
building = ['log_sqm','yearbuilt_norm']
usage    = sorted([c for c in df.columns if c.startswith('usage_')])

groups = {
    'ID (not used in model)' : id_cols,
    'TARGET'                 : target,
    'Calendar'               : calendar,
    'Lag / Rolling'          : lag_roll,
    'Weather'                : weather,
    'Building'               : building,
    'Usage Dummies'          : usage,
}

total_features = 0
for group, cols in groups.items():
    if group not in ['ID (not used in model)', 'TARGET']:
        total_features += len(cols)
    nan_flag = any(df[c].isna().sum() > 0 for c in cols if c in df.columns)
    print(f'  [{len(cols):2d}]  {group}{"  ← has NaN" if nan_flag else ""}')
    for c in cols:
        print(f'          {c}')
    print()

print(f'Total model features : {total_features}')
print(f'Total rows           : {len(df):,}')
print(f'Buildings            : {df["building_id"].nunique():,}')

## Phase 5 Summary

| Item | Value |
|---|---|
| Input | `data/processed/daily_energy_dataset.csv` (1,004,892 rows) |
| Warm-up rows dropped | 10,080 |
| **Final rows** | **994,812** |
| Total features | **37** |
| Buildings | 1,440 |
| Date range | 2016-01-08 → 2017-12-31 |
| Leakage check | ✅ PASSED |
| Output (CSV) | `data/processed/features_dataset.csv` (237.65 MB) |
| Output (Parquet) | `data/processed/features_dataset.parquet` (37.35 MB) |

✅ **Phase 5 complete. Awaiting confirmation to proceed to Phase 6.**